<a href="https://colab.research.google.com/github/andreslill/mosquito-season-risk/blob/main/mosquito_risk_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ERA5 Pre-computation of Mosquito Season Risk Pipeline


**Project:** When is mosquito season in your city?  
**Author:** Andrés Lill · March 2026  
**Data:** ERA5 monthly means, 1991–2020 (Copernicus CDS)  
**Output:** `mosquito_risk_cities.csv` (1,421 cities × 12 months)

## Overview

Monthly climate suitability scores (0–1) are computed for two Aedes mosquito species across 1,421 cities worldwide.

**Species modelled:**
- *Ae. aegypti*: principal vector for dengue, Zika, chikungunya, and yellow fever;
  highly efficient transmission, tropical/subtropical distribution
  (Mordecai et al. 2017)
- *Ae. albopictus*: competent vector for the same arboviruses. Lower transmission
  efficiency for dengue and Zika but primary vector for chikungunya in temperate
  regions. Broader thermal tolerance and invasive range expansion into temperate
  regions including large parts of Europe
  (Bonizzoni et al. 2013, Tegar et al. 2026, ECDC Vector Maps)

**Suitability formula:**

```
Risk Score (aegypti)    = TempScore × VPDScore
Risk Score (albopictus) = TempScore × VPDScore × PhotoFactor*
```
*PhotoFactor applied only outside the tropics (|lat| ≥ 23.5°)

**Important:** Scores reflect *climate suitability*, not confirmed mosquito presence, disease risk, or actual population abundance.


## Notebook structure

| Section | Description |
|---|---|
| 1. Setup | Install dependencies, authenticate to Copernicus CDS |
| 2. Download ERA5 data | Download 30-year monthly means via CDS API |
| 3. Inspect ERA5 files | Verify downloaded NetCDF structure |
| 4. Compute risk scores | Main pipeline: sample ERA5 → compute scores → export CSV |

## Data sources

| Dataset | Source |
|---|---|
| Climate normals | ERA5 monthly means, Copernicus CDS (1991–2020) |
| City list | SimpleMaps World Cities Basic v1.901 (pop ≥ 500,000) |

## Model Parameter

| Model parameter | Source |
|---|---|
| Temperature suitability | Doeurk et al. 2025 (*Parasites & Vectors*) |
| VPD linearization | Schmidt et al. 2018 (*Parasites & Vectors*) |
| Photoperiod gate | Lacour et al. 2015 (*PLOS ONE*) |

## 1. Setup

### 1.1 Authenticate to Copernicus CDS

Reads your CDS API key from Colab Secrets (key name: `CDSAPI_KEY`) and writes the required `.cdsapirc` config file.

To get a CDS API key: register at [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu), then find your key under your profile.

In [4]:
from google.colab import userdata
import os, pathlib

# Read API key from Colab Secrets (never hardcode credentials)
key = userdata.get("CDSAPI_KEY")
assert key, "CDSAPI_KEY not found in Colab Secrets — add it under Tools > Secrets"

# Write the CDS API config file required by the cdsapi client
pathlib.Path("/root/.cdsapirc").write_text(
    "url: https://cds.climate.copernicus.eu/api\n"
    f"key: {key}\n"
)

CDS credentials written to /root/.cdsapirc


### 1.2 Test CDS connection

Verify that the CDS client can connect successfully using the configured API key.

In [2]:
!pip install cdsapi

import cdsapi
c = cdsapi.Client()
print("Connected")

Connected


## 2. Download ERA5 Data

Downloads ERA5 monthly averaged reanalysis for the 1991–2020 WMO reference period.

**Variables downloaded:**
- `2m_temperature` (K) used to derive temperature suitability
- `2m_dewpoint_temperature` (K) used with temperature to derive RH (%) and VPD (kPa)
- `total_precipitation` (m/day), included as contextual information only; not part of the risk score formula

**Why monthly means?**  
We use a "typical year" climatology (30-year average per calendar month) rather than individual years, so that scores reflect long-term climate conditions rather than year-to-year variability.

**Note on units:**  
Temperature and dewpoint are returned in Kelvin  (convert with `K - 273.15`).Precipitation is returned in m/day (convert to mm/month with `× 1000 × days_in_month`)

> *Climate data based on the 1991–2020 reference period. Recent warming trends may shift actual season windows beyond what this model shows.*



In [6]:
import os
import cdsapi

# Update the paths below before running this notebook.
# Set your output path: Adjust to your local environment or Google Drive mount point
OUTPUT_PATH = "era5_monthly_1991_2020.nc"
os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)

c = cdsapi.Client()
c.retrieve(
    "reanalysis-era5-single-levels-monthly-means",
    {
        "product_type": "monthly_averaged_reanalysis",
        "variable": [
            "2m_temperature",
            "2m_dewpoint_temperature",
            "total_precipitation",
        ],
        "year": [str(y) for y in range(1991, 2021)],
        "month": ["01","02","03","04","05","06","07","08","09","10","11","12"],
        "time": "00:00",
        "format": "netcdf",
    },
    OUTPUT_PATH,
)
print("Download complete.")

2026-03-16 08:51:14,154 INFO Request ID is 8711a086-d90c-40be-9093-cfed7ce59491
INFO:ecmwf.datastores.legacy_client:Request ID is 8711a086-d90c-40be-9093-cfed7ce59491
2026-03-16 08:51:14,311 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-03-16 08:51:47,553 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-03-16 08:52:04,780 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


44f4d30ba431fe534cf14e058df6a4cc.zip:   0%|          | 0.00/1.36G [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Mosquito Project/Data/era5_monthly_1991_2020_.nc'

## 3. Inspect ERA5 Files

The CDS download produces a ZIP archive containing two NetCDF files with different stream types:
- `avgua` - instantaneous fields: 2m temperature (`t2m`) and 2m dewpoint (`d2m`), in Kelvin
- `avgad` - accumulated fields: total precipitation (`tp`), typically in m/day

We inspect both files to verify their structure before running the main pipeline.


In [ ]:
import xarray as xr

# Update the paths below before running this notebook.
# Set your ERA5 extract directory: adjust to your local environment
EXTRACT_DIR = "./"  # e.g. "/content/drive/MyDrive/Mosquito Project/Data/"

# Load both NetCDF files
ds1 = xr.open_dataset(EXTRACT_DIR + "data_stream-moda_stepType-avgua.nc")  # t2m, d2m
ds2 = xr.open_dataset(EXTRACT_DIR + "data_stream-moda_stepType-avgad.nc")  # tp

print("=== ds1 (avgua): temperature + dewpoint ===")
print(ds1)
print("\n=== ds2 (avgad): precipitation ===")
print(ds2)

## 4. Compute Risk Scores

Main pipeline: samples ERA5 climatology for each city, computes suitability scores, and exports a Tableau-ready CSV.

### Suitability model summary

**Temperature suitability (`TempScore`)**  
Triangular thermal performance curve: 0 at Tmin and Tmax, 1 at Topt, linear between.  
Parameters from Doeurk et al. 2025 (adult survival curve):

| Species | Tmin (°C) | Topt (°C) | Tmax (°C) |
|---|---|---|---|
| *Ae. aegypti* | 14.97 | 27.1 | 39.15 |
| *Ae. albopictus* | 11.02 | 24.5 | 38.07 |

**Desiccation stress (`VPDScore`)**  
Linearized vapour pressure deficit suitability (Schmidt et al. 2018):
- VPD ≤ 1.0 kPa: score = 1.0
- VPD ≥ 3.0 kPa: score = 0.0
- Linear between

VPD is derived from ERA5 temperature and dewpoint using the Magnus (Tetens) approximation.

**Photoperiod gate (`PhotoFactor` — *Ae. albopictus* only)**  
Applied **only outside the tropics** (|lat| ≥ 23.5°). In tropical latitudes, daylength is near-constant year-round and is not a meaningful diapause cue, so PhotoFactor = 1.0 there.  
Thresholds from Lacour et al. 2015:
- Daylength < 11.25h → 0.0 (diapause)
- 11.25–13.5h → 0.5 (transition)
- ≥ 13.5h → 1.0 (active)

**Precipitation**  
Included as contextual information only. Does not contribute to the suitability score.


In [ ]:
#!/usr/bin/env python3
"""
Mosquito Climate Suitability — ERA5 Pre-computation (1991–2020)
===============================================================

Purpose
-------
Generate a Tableau-ready CSV (city × month) with climate-based suitability
scores for Ae. aegypti and Ae. albopictus.

This model estimates *climate suitability for mosquito activity*, NOT disease
risk, NOT confirmed presence, and NOT mosquito abundance.

Climate baseline
----------------
ERA5 monthly means (Copernicus CDS) averaged over the 1991–2020 WMO
climatological reference period ("typical year" climatology).

> Climate data based on the 1991–2020 reference period. Recent warming
> trends may shift actual season windows beyond what this model shows.

Key methodological choices
--------------------------
1. Temperature suitability: triangular curve (0 at Tmin/Tmax, 1 at Topt).
   Parameters from Doeurk et al. 2025 (adult survival curves).

2. Desiccation stress: linearized VPD score (Schmidt et al. 2018):
   - VPD_score = 1.0  at VPD ≤ 1.0 kPa
   - VPD_score = 0.0  at VPD ≥ 3.0 kPa
   - linear between

3. Photoperiod (Ae. albopictus only, Lacour et al. 2015):
   Applied ONLY outside the tropics (|lat| ≥ 23.5°). In tropical/subtropical
   regions daylength varies little year-round and applying the temperate gate
   creates systematic artefacts — photoperiod factor is set to 1.0 there.

4. Precipitation: included as contextual information only; not part of the
   risk score formula. Classified relative to each city's own annual range
   (within-city quartiles) to avoid arbitrary global mm thresholds.

Outputs
-------
One row per city per month (12 rows per city) with:
- Climate normals (Temp, RH, VPD, precip, photoperiod)
- Component scores (temp_score_*, vpd_score, photo_factor_albopictus)
- Final suitability scores: risk_score_aegypti, risk_score_albopictus  [0–1]
- Contextual: precip_context (low / moderate / high / very_high)
"""


import numpy as np
import pandas as pd
import xarray as xr
import math
from tqdm import tqdm

# ── CONFIG ────────────────────────────────────────────────────────
# Update the paths below before running this notebook.
# Adjust all paths to your local environment or Google Drive mount point
WORLDCITIES_CSV = "worldcities.csv"          # SimpleMaps World Cities Basic v1.901
EXTRACT_DIR     = "./"                        # directory containing the ERA5 NetCDF files
OUTPUT_CSV      = "06_mosquito_risk_cities_aedes_only.csv"
MIN_POPULATION  = 500_000

# ── SPECIES TEMPERATURE PARAMETERS (Tmin, Topt, Tmax in °C) ──────
# Source: Doeurk et al. 2025 — adult survival curve parameters
TEMP_PARAMS = {
    "aegypti":    (14.97, 27.1,  39.15),
    "albopictus": (11.02, 24.5,  38.07),
}

# ── VPD THRESHOLDS (kPa) — linearized from Schmidt et al. 2018 ───
VPD_LOW  = 1.0   # VPD score = 1.0 at or below this threshold
VPD_HIGH = 3.0   # VPD score = 0.0 at or above this threshold

# ── PHOTOPERIOD THRESHOLDS (hours) — Lacour et al. 2015 ──────────
PHOTO_LOW  = 11.25   # minimum daylength for spring egg hatching
PHOTO_HIGH = 13.5    # critical photoperiod for diapause induction

# ── TROPICS BOUNDARY ──────────────────────────────────────────────
# Photoperiod gate applied only outside the tropics (|lat| ≥ 23.5°)
TROPICS_LAT = 23.5


# ── HELPER FUNCTIONS ──────────────────────────────────────────────

def temp_score_triangle(temp_c: float, tmin: float, topt: float, tmax: float) -> float:
    """
    Triangular thermal performance curve.
    Returns 0 at Tmin/Tmax, 1 at Topt, linear between.
    """
    if temp_c <= tmin or temp_c >= tmax:
        return 0.0
    if temp_c <= topt:
        return (temp_c - tmin) / (topt - tmin)
    return (tmax - temp_c) / (tmax - topt)


def calc_vpd_kpa(temp_c: float, rh_pct: float) -> float:
    """
    Compute vapour pressure deficit (VPD) in kPa.
    Uses the Tetens (Magnus) approximation for saturation vapour pressure.
    """
    svp = 0.6108 * math.exp((17.27 * temp_c) / (temp_c + 237.3))
    return max(0.0, svp * (1.0 - rh_pct / 100.0))


def vpd_score(vpd_kpa: float) -> float:
    """
    Linearized VPD suitability score (Schmidt et al. 2018).
    1.0 below VPD_LOW, 0.0 above VPD_HIGH, linear between.
    """
    if vpd_kpa <= VPD_LOW:
        return 1.0
    if vpd_kpa >= VPD_HIGH:
        return 0.0
    return (VPD_HIGH - vpd_kpa) / (VPD_HIGH - VPD_LOW)


def photoperiod_hours(lat_deg: float, month: int) -> float:
    """
    Approximate daylength (hours) for a given latitude and month.
    Uses the 15th day of each month as a representative midpoint.
    """
    doy_mid = [15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349]
    doy = doy_mid[month - 1]
    lat = math.radians(lat_deg)
    decl = math.radians(23.44) * math.sin(math.radians(360 / 365.0 * (doy - 81)))
    cos_ha = max(-1.0, min(1.0, -math.tan(lat) * math.tan(decl)))
    return round(2 * math.degrees(math.acos(cos_ha)) / 15.0, 2)


def albopictus_photo_factor(daylen_h: float, lat_deg: float) -> float:
    """
    Ae. albopictus photoperiod gate — applied only outside the tropics.

    In temperate regions, photoperiod is a strong cue for diapause-related
    seasonal behaviour (Lacour et al. 2015). In tropical/subtropical regions
    (|lat| < 23.5°), daylength varies little year-round and is not a meaningful
    diapause cue; applying the temperate threshold creates systematic artefacts.

    Returns:
        1.0  if tropical (no photoperiod penalty)
        0.0  if daylen < 11.25 h  (diapause)
        0.5  if 11.25 ≤ daylen < 13.5 h  (transition)
        1.0  if daylen ≥ 13.5 h  (fully active)
    """
    if abs(lat_deg) < TROPICS_LAT:
        return 1.0
    if daylen_h < PHOTO_LOW:
        return 0.0
    if daylen_h < PHOTO_HIGH:
        return 0.5
    return 1.0


def days_in_month(month: int) -> int:
    """Days per calendar month (non-leap year). Used for mm/month conversion."""
    return [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][month - 1]


def precip_context(monthly_precip_series: pd.Series) -> list:
    """
    Classify each month's precipitation relative to the city's own annual range.
    Uses within-city quartiles to avoid arbitrary global mm thresholds.
    Returns a list of labels: 'low', 'moderate', 'high', 'very_high'.
    """
    q25, q50, q75 = monthly_precip_series.quantile([0.25, 0.50, 0.75])
    cats = []
    for v in monthly_precip_series:
        if v <= q25:
            cats.append("low")
        elif v <= q50:
            cats.append("moderate")
        elif v <= q75:
            cats.append("high")
        else:
            cats.append("very_high")
    return cats


# ── LOAD ERA5 CLIMATOLOGY ─────────────────────────────────────────
print("Loading ERA5 datasets...")
ds1 = xr.open_dataset(EXTRACT_DIR + "data_stream-moda_stepType-avgua.nc")  # t2m, d2m (Kelvin)
ds2 = xr.open_dataset(EXTRACT_DIR + "data_stream-moda_stepType-avgad.nc")  # tp (m/day)

print("Computing 30-year monthly climatology (1991–2020)...")
t2m_clim = ds1["t2m"].groupby("valid_time.month").mean("valid_time")  # (12, lat, lon) Kelvin
d2m_clim = ds1["d2m"].groupby("valid_time.month").mean("valid_time")  # (12, lat, lon) Kelvin
tp_clim  = ds2["tp"].groupby("valid_time.month").mean("valid_time")   # (12, lat, lon) m/day


# ── LOAD AND FILTER CITIES ────────────────────────────────────────
print("Loading city list...")
cities = pd.read_csv(WORLDCITIES_CSV, usecols=["city", "country", "lat", "lng", "population"])
cities = cities.rename(columns={"lng": "lon"})
cities["population"] = pd.to_numeric(cities["population"], errors="coerce")
cities = cities.dropna(subset=["population", "lat", "lon"])
cities = cities[cities["population"] >= MIN_POPULATION]
cities = cities.drop_duplicates(subset=["city", "country"])
cities = cities.sort_values("population", ascending=False).reset_index(drop=True)
print(f"  {len(cities)} cities loaded (population ≥ {MIN_POPULATION:,})")


# ── MAIN SAMPLING LOOP ────────────────────────────────────────────
print("Sampling ERA5 climatology and computing risk scores...")
rows = []

for _, city in tqdm(cities.iterrows(), total=len(cities), desc="Cities"):
    lat, lon = float(city["lat"]), float(city["lon"])

    # ERA5 uses 0–360 longitude convention; convert negative (western) values
    lon360 = lon if lon >= 0 else lon + 360

    # Sample nearest ERA5 grid point for all 12 months at once
    t2m = t2m_clim.sel(latitude=lat, longitude=lon360, method="nearest").values  # (12,) Kelvin
    d2m = d2m_clim.sel(latitude=lat, longitude=lon360, method="nearest").values  # (12,) Kelvin
    tp  = tp_clim.sel(latitude=lat,  longitude=lon360, method="nearest").values  # (12,) m/day

    # Unit conversions
    temp_c = t2m - 273.15  # K → °C
    dewp_c = d2m - 273.15  # K → °C

    # Relative humidity (%) from temperature and dewpoint (Magnus approximation)
    rh = 100 * np.exp((17.27 * dewp_c) / (dewp_c + 237.3)) / \
               np.exp((17.27 * temp_c) / (temp_c + 237.3))
    rh = np.clip(rh, 0, 100)

    # Precipitation: m/day → mm/month (using exact days per month)
    precip_mm_month = np.array([float(tp[i]) * 1000 * days_in_month(i + 1) for i in range(12)])

    # Classify precipitation relative to city's own annual distribution
    precip_cats = precip_context(pd.Series(precip_mm_month))

    for i, month in enumerate(range(1, 13)):
        t   = float(temp_c[i])
        rh_ = float(rh[i])
        p   = float(precip_mm_month[i])

        vpd    = calc_vpd_kpa(t, rh_)
        vs     = vpd_score(vpd)
        daylen = photoperiod_hours(lat, month)

        ts_aeg = temp_score_triangle(t, *TEMP_PARAMS["aegypti"])
        ts_alb = temp_score_triangle(t, *TEMP_PARAMS["albopictus"])
        pf_alb = albopictus_photo_factor(daylen, lat)

        rows.append({
            # City metadata
            "city":                    city["city"],
            "country":                 city["country"],
            "lat":                     round(lat, 4),
            "lon":                     round(lon, 4),
            "population":              int(city["population"]),
            "month":                   month,

            # Climate normals (ERA5, 1991–2020)
            "temp_mean_c":             round(t, 2),
            "rh_mean_pct":             round(rh_, 1),
            "precip_sum_mm":           round(p, 1),
            "vpd_kpa":                 round(vpd, 4),
            "photoperiod_h":           daylen,

            # Suitability components
            "vpd_score":               round(vs, 4),
            "temp_score_aegypti":      round(ts_aeg, 4),
            "temp_score_albopictus":   round(ts_alb, 4),
            "photo_factor_albopictus": pf_alb,

            # Final suitability scores (0–1)
            "risk_score_aegypti":      round(ts_aeg * vs, 4),
            "risk_score_albopictus":   round(ts_alb * vs * pf_alb, 4),

            # Contextual only — not used in the risk formula
            "precip_context":          precip_cats[i],
        })


# ── EXPORT ────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\nDone! {len(cities)} cities × 12 months = {len(df):,} rows")
print(f"Saved to: {OUTPUT_CSV}")
print("\nSample output (first 12 rows):")
print(df.head(12).to_string(index=False))


### 4.1 Inspect output

Quick sanity checks on the exported CSV.

In [ ]:
import pandas as pd

# Update the paths below before running this notebook.
# Set your output path: Adjust to your local environment or Google Drive mount point
OUTPUT_CSV = "06_mosquito_risk_cities_aedes_only.csv"

df = pd.read_csv(OUTPUT_CSV)

print(f"Shape: {df.shape}  ({df['city'].nunique()} cities × 12 months)")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nRisk score ranges:")
print(f"  aegypti:    {df['risk_score_aegypti'].min():.3f} – {df['risk_score_aegypti'].max():.3f}")
print(f"  albopictus: {df['risk_score_albopictus'].min():.3f} – {df['risk_score_albopictus'].max():.3f}")
print(f"\nSample — Hamburg, Germany:")
print(df[df['city'] == 'Hamburg'].to_string(index=False))